# APARCH e Component-GARCH

Neste notebook, exploramos dois modelos GARCH avancados:

- **APARCH** (Asymmetric Power ARCH) de Ding, Granger e Engle (1993)
- **IGARCH** (Integrated GARCH) de Engle e Bollerslev (1986)
- **Component-GARCH** de Engle e Lee (1999)

Esses modelos generalizam o GARCH padrao em direcoes diferentes:
o APARCH introduz um **parametro de potencia** flexivel, o IGARCH impoe
**persistencia unitaria**, e o Component-GARCH decompoe a volatilidade em
**componentes de curto e longo prazo**.

**Conteudo:**
1. APARCH de Ding, Granger e Engle (1993)
2. O parametro de potencia delta
3. IGARCH - Integrated GARCH
4. Component-GARCH de Engle e Lee (1999)
5. Componente transitorio vs permanente
6. Aplicacao: dados do Bitcoin

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. APARCH de Ding, Granger e Engle (1993)

O modelo **APARCH** (Asymmetric Power ARCH) e uma generalizacao poderosa que unifica
varios modelos GARCH atraves de um **parametro de potencia** $\delta$:

$$\sigma_t^{\delta} = \omega + \alpha (|\epsilon_{t-1}| - \gamma \epsilon_{t-1})^{\delta} + \beta \sigma_{t-1}^{\delta}$$

onde:
- $\delta > 0$: parametro de potencia (estimado ou fixo)
- $\gamma \in (-1, 1)$: parametro de assimetria (leverage)
- Se $\gamma > 0$: choques negativos tem maior impacto

**Casos especiais:**
| $\delta$ | $\gamma$ | Modelo equivalente |
|----------|----------|--------------------|
| 2 | 0 | GARCH |
| 2 | $\neq 0$ | GJR-GARCH |
| 1 | 0 | AVGARCH (Taylor, 1986) |
| 1 | $\neq 0$ | TARCH (Zakoian, 1994) |

In [ ]:
# Carregar dados do S&P 500
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

# TODO: Estime um APARCH(1,1) com archbox
# Dicas:
# - model_aparch = APARCH(returns.values, p=1, q=1)
# - results_aparch = model_aparch.fit()
# - print(results_aparch.summary())
# - Observe o valor estimado de delta e gamma

## 2. O parametro de potencia $\delta$

Ding, Granger e Engle (1993) mostraram empiricamente que a autocorrelacao de $|r_t|^d$
e **maximizada** quando $d \approx 1$ (nao $d = 2$ como no GARCH padrao).

Isso sugere que modelar a **volatilidade** (desvio padrao, $\delta = 1$) pode ser mais
apropriado do que modelar a **variancia** ($\delta = 2$).

O APARCH permite que os dados determinem o valor otimo de $\delta$:
- $\delta = 2$: equivale ao GARCH padrao (modela variancia)
- $\delta = 1$: modela o desvio padrao (AVGARCH)
- $\delta$ livre: o valor otimo e determinado pelos dados

Vamos comparar modelos com diferentes valores fixos de $\delta$.

In [ ]:
# TODO: Compare APARCH com diferentes valores fixos de delta
# Dicas:
# - Estime APARCH com delta livre (padrao)
# - Compare o delta estimado com delta=1 e delta=2
# - Compare AIC/BIC dos modelos
# - O delta estimado esta proximo de 1 ou 2?

## 3. IGARCH - Integrated GARCH

O **IGARCH** (Integrated GARCH) de Engle e Bollerslev (1986) e um caso especial
do GARCH(1,1) onde a **persistencia e unitaria**:

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + (1 - \alpha) \sigma_{t-1}^2$$

ou seja, $\alpha + \beta = 1$ (com $\beta = 1 - \alpha$).

**Implicacoes:**
- A variancia incondicional **nao existe** (nao e finita)
- Choques na volatilidade tem efeito **permanente** (nunca se dissipam completamente)
- A previsao de longo prazo **nao converge** para um nivel fixo
- Comum em dados financeiros de alta frequencia

O IGARCH e util quando o GARCH estimado tem persistencia muito proxima de 1.

In [ ]:
# TODO: Estime IGARCH(1,1) com restricao alpha+beta=1
# Dicas:
# - model_igarch = IGARCH(returns.values)
# - results_igarch = model_igarch.fit()
# - print(results_igarch.summary())
# - Verifique que persistence() = 1.0 (ou muito proximo)
# - Compare a volatilidade condicional com o GARCH(1,1)

## 4. Component-GARCH de Engle e Lee (1999)

O **Component-GARCH** (CGARCH) decompoe a variancia condicional em dois componentes:

**Componente permanente (tendencia de longo prazo):**
$$q_t = \omega + \rho (q_{t-1} - \omega) + \phi (\epsilon_{t-1}^2 - \sigma_{t-1}^2)$$

**Componente transitorio (desvios de curto prazo):**
$$\sigma_t^2 - q_t = \alpha (\epsilon_{t-1}^2 - q_{t-1}) + \beta (\sigma_{t-1}^2 - q_{t-1})$$

**Variancia condicional total:**
$$\sigma_t^2 = q_t + (\text{componente transitorio})$$

**Interpretacao:**
- $q_t$: nivel de **longo prazo** da volatilidade (varia lentamente)
- $\sigma_t^2 - q_t$: desvio de **curto prazo** (decai rapidamente)
- $\rho$: persistencia do componente permanente (proximo de 1)
- $\alpha + \beta$: persistencia do componente transitorio (menor que $\rho$)

In [ ]:
# TODO: Estime Component-GARCH e visualize os dois componentes
# Dicas:
# - model_cgarch = ComponentGARCH(returns.values)
# - results_cgarch = model_cgarch.fit()
# - print(results_cgarch.summary())
# - A volatilidade condicional total: results_cgarch.conditional_volatility

## 5. Componente transitorio vs permanente

A decomposicao em componentes oferece **interpretacao economica** rica:

- O **componente permanente** ($q_t$) captura mudancas estruturais no nivel de risco
  do mercado (crises, mudancas de regime, tendencias macroeconomicas)
- O **componente transitorio** ($\sigma_t^2 - q_t$) captura choques de curto prazo
  (noticias, eventos, etc.) que se dissipam rapidamente

Essa decomposicao e util para:
- Distinguir entre risco **sistematico** (permanente) e **idiossincratico** (transitorio)
- Definir horizontes de investimento: para horizontes longos, foque no componente permanente
- Detectar **mudancas de regime** no nivel base de volatilidade

In [ ]:
# TODO: Plote ambos componentes de volatilidade separadamente
# Dicas:
# - Acesse a volatilidade total: results_cgarch.conditional_volatility
# - Crie um grafico com 2 paineis (fig, axes = plt.subplots(2, 1))
# - Painel 1: componente permanente (longo prazo)
# - Painel 2: componente transitorio (curto prazo)
# - Compare as escalas e a dinamica dos dois componentes

## 6. Aplicacao: dados do Bitcoin

O **Bitcoin** apresenta caracteristicas unicas em relacao a ativos tradicionais:

- Volatilidade **muito mais alta** que acoes
- Possiveis **mudancas de regime** na volatilidade (ciclos de bull/bear)
- Caudas **extremamente pesadas**
- Efeito alavancagem potencialmente **diferente** (ou ausente)

O APARCH (com potencia flexivel) e o Component-GARCH (decomposicao em tendencia + ciclo)
sao particularmente interessantes para criptomoedas.

In [ ]:
# TODO: Estime APARCH e Component-GARCH nos dados bitcoin_returns.csv
# Dicas:
# - btc = pd.read_csv('../data/bitcoin_returns.csv', parse_dates=['date'], index_col='date')
# - btc_returns = btc['returns']
# - Estime APARCH e ComponentGARCH
# - O delta estimado do APARCH e diferente do S&P 500?
# - O componente permanente do CGARCH e mais variavel no Bitcoin?
# - Compare os criterios de informacao

## Conclusao

Neste notebook, aprendemos:

- O modelo **APARCH** e sua generalizacao via parametro de potencia $\delta$
- Que $\delta \approx 1$ frequentemente ajusta melhor que $\delta = 2$ (GARCH padrao)
- O **IGARCH** e suas implicacoes de persistencia unitaria
- O **Component-GARCH** e a decomposicao em tendencia de longo prazo + ciclo de curto prazo
- Aplicacao pratica em dados de criptomoedas

No proximo notebook, faremos uma **comparacao sistematica** de todos os modelos
GARCH univariados, incluindo diagnosticos, previsao out-of-sample e ranking final.